<a href="https://colab.research.google.com/github/ripims1/COMP-3608-PROJECT/blob/tyler-data-blending/Model_Ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
# Global library declaration
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import numpy as np

In [34]:
# ========================
#  Load and Prepping Data
# ========================

#Load the data
train_df = pd.read_csv('train_processed.csv')
test_df = pd.read_csv('test_processed.csv')

#Separate Features (X) and Target (y)
X_train_raw = train_df.drop('Churn', axis=1).values
y_train_raw = train_df['Churn'].values
X_test_raw = test_df.values

# Splitting training data to use for training and evaluation
X_train_main, X_val, y_train_main, y_val = train_test_split(
    X_train_raw, y_train_raw, test_size=0.2, random_state=42
)


In [36]:
# ==========
#  LightGBM
# ==========

# Use same scaled datasets as in the NN model
X_train_lgb = X_train_main
Y_train_lgb = y_train_main
X_val_lgb = X_val
X_test_lgb = X_test_raw

# Convert the training and validation data to the LightGBM datset format
train_data = lgb.Dataset(X_train_lgb, label=Y_train_lgb)
val_data = lgb.Dataset(X_val_lgb, label=y_val)

# Define the model parametrs for LightGBM to control how it learns
params = {
    'objective': 'binary',        # Binary Classification problem
    'metric': 'auc',              # Area under ROC curve
    'boosting_type': 'gbdt',      # Gradient Boosting decision trees
    'learning_rate': 0.03,        # Learning step size
    'num_leaves': 64,             # Controls complexity of trees
    'feature_fraction': 0.8,      # Randomly select features to prevent overfitting
    'bagging_fraction': 0.8,      # Randomly sample data
    'bagging_freq': 5,            # Frequency of bagging
    'verbosity': -1               # Suppress training output
}

# Train the model using the dataset
lgb_model = lgb.train(params,                               # All model parameters
                      train_data,                           # Training Dataset
                      num_boost_round=1000,                 # Number of trees to build
                      valid_sets=[val_data],                # Validation dataset to evalulate during training
                      callbacks=[lgb.early_stopping(50)]    # Stop training if validation auc doesnt improve for 50 rounds
                      )

# Generate probability predictions for the validation dataset
lgb_val_preds = lgb_model.predict(X_val_lgb)

# Generate probability predictions for the test dataset and output between 0 and 1
prediction_LGBM = lgb_model.predict(X_test_lgb)

print("Successfully generated prediction")

# Retrain LightGBM on FULL dataset (no split now)
final_lgb_model = lgb.train(
    params,
    lgb.Dataset(X_train_raw, label=y_train_raw),
    num_boost_round=lgb_model.best_iteration   # use best iteration from earlier
)

# Generate predictions on test data using FULL model
final_prediction = final_lgb_model.predict(X_test_raw)


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[624]	valid_0's auc: 0.91668
Successfully generated prediction


In [37]:
# ==================
#  FINAL SUBMISSION
# ==================

start_id = 594194
end_id = 848848

submission_df = pd.DataFrame({
    'id': range(start_id, end_id + 1),
    'Churn': final_prediction
})

submission_df.to_csv('submission8.csv', index=False)

print("Blended submission file created!")

Blended submission file created!
